# Meta-learning N_SHOT sweep — EfficientNet-B0 (dataset privado)

Compara configuraciones de **meta-learning** (barriendo `N_SHOT`) para **EfficientNet-B0 + meta**
sobre el **dataset privado** de glaucoma. **Evaluación NO episódica (opción A)**, igual que el
experimento de selección de backbone: F1-macro sobre el **val completo**.

- **N_WAY = 2** · **N_SHOT ∈ {3,6,9,12,15,18,21}** · `N_QUERY`, `N_TRAINING_EPISODES` configurables.
- Por cada `N_SHOT`: **meta-entreno** EfficientNet-B0 con episodios `n_shot=N_SHOT` (desde *train*) →
  armo los 2 prototipos con un soporte balanceado N-shot de *train* → **clasifico el val completo** (F1-macro).
- **IoU Grad-CAM** sobre el *val* (máscaras = `disc ∪ cup` de los `.npy`; clase glaucoma, opción Y).
- **Score** por `N_SHOT`: `0.40·F1 + 0.40·IoU + 0.20·(1 − VRAM_norm)`. Un solo backbone → VRAM constante.
- Guarda `.pth` + `meta_nshot_summary.json` en **Drive**.

Límite de feasibilidad = **train (minoritaria 36)**: el *val* no restringe `N_SHOT` (se clasifica entero).

## Celda 0: Bootstrap (repo + EasyFSL + Drive)

In [ ]:
# --- Bootstrap Colab: repo + easyfsl + Drive ---
import os
REPO = "/content/Medgemma_Segmentation_CIARP_2026"
if not os.path.isdir(REPO):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b Classifier-Implementation https://github.com/TheBug95/Medgemma_Segmentation_CIARP_2026.git {REPO}
%cd {REPO}
!GIT_LFS_SKIP_SMUDGE=1 git pull origin Classifier-Implementation
!pip install -q easyfsl
from google.colab import drive
drive.mount('/content/drive')
import torch; print("GPU:", torch.cuda.is_available())

## Celda 1: Config + imports + helpers

**Ajusta** `DATASET_DIR` (carpeta con imágenes+máscaras juntas), `SPLIT_JSON` y `RESULTS_DIR` de tu Drive.

In [ ]:
import sys, os, json, gc, datetime
import numpy as np
import torch

REPO = "/content/Medgemma_Segmentation_CIARP_2026"
sys.path.insert(0, f"{REPO}/Experiments/Classifier_Selection")   # modules/, scripts/
sys.path.insert(0, f"{REPO}/Classifier")                          # private_data_module

from modules.prototype_classifier import PrototypeClassifier
from modules.cnn_classifier import CNNClassifier
from scripts.few_shot import build_support_indices, create_few_shot_loader
from scripts.evaluate import evaluate_classification
from scripts.benchmark_inference import run_benchmark
from private_data_module import PrivateDataModule

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============== CONFIG — AJUSTA ESTAS RUTAS DE TU DRIVE ==============
DATASET_DIR = "/content/drive/MyDrive/Todo Dataset"                     # TODO: imagenes .jpg + mascaras .npy (misma carpeta)
SPLIT_JSON  = "/content/drive/MyDrive/Todo Dataset/split.json"         # TODO: tu split.json
RESULTS_DIR = "/content/drive/MyDrive/results_meta_efficientNet" # .pth + summary se guardan aqui

# ---- Barrido y params (modificables) ----
N_SHOTS             = [3, 6, 9, 12, 15, 18, 21]
N_QUERY             = 10     # query por clase en los episodios de META-TRAINING (opcion A no tiene query de evaluacion)
N_TRAINING_EPISODES = 100    # nro de episodios de meta-training (muestreados del train)
SEEDS               = [42, 123, 456, 789, 1024]
IMAGE_SIZE          = [224, 224]
META_LR             = 0.001
# ====================================================================

N_KEYS = [f"N{n}" for n in N_SHOTS]
BACKBONE = "efficientnet_b0"
BATCH_SIZE = 16
PERCENTILE = 95

# ---- IoU/pointing del Grad-CAM (clase glaucoma) vs mascara GT del disco ----
def _iou(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    u = np.logical_or(pred, gt).sum()
    return float(np.logical_and(pred, gt).sum() / u) if u else 0.0

def _pointing_hit(heatmap, gt):
    y, x = np.unravel_index(int(np.argmax(heatmap)), heatmap.shape)
    return 1.0 if gt[y, x] > 0 else 0.0

def evaluate_gradcam_quality(model, data_loader):
    glaucoma_idx = model.class_names.index("glaucoma") if "glaucoma" in model.class_names else 1
    ious, pts = [], []
    for batch in data_loader:
        images, masks = batch["image"], batch["mask"]
        for i in range(images.size(0)):
            gt = masks[i, 0].numpy()
            hm = model.get_gradcam(images[i], target_class=glaucoma_idx)
            thr = np.percentile(hm, PERCENTILE)
            ious.append(_iou((hm >= thr).astype(np.float32), gt))
            pts.append(_pointing_hit(hm, gt))
    return {"mean_iou": float(np.mean(ious)) if ious else 0.0,
            "mean_pointing_accuracy": float(np.mean(pts)) if pts else 0.0}

print("Config OK | N_SHOTS =", N_SHOTS, "| N_QUERY(meta) =", N_QUERY, "| episodios meta =", N_TRAINING_EPISODES)

## Celda 2: DataModule del dataset privado

Imágenes y máscaras están en la **misma carpeta** (`DATASET_DIR`). Verifica que el barrido entra en *train*
(de ahí salen el soporte balanceado y los episodios de meta-training).

In [ ]:
data_module = PrivateDataModule({
    "split_json": SPLIT_JSON,
    "images_dir": DATASET_DIR,   # imagenes y mascaras en la MISMA carpeta
    "masks_dir":  DATASET_DIR,
    "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
    "seed": 42, "cache_images": True,
})
val_loader = data_module.get_val_loader()
print("Distribucion por split:", data_module.get_class_distribution())

g_train = len(data_module.get_glaucoma_indices("train"))
min_train = min(g_train, len(data_module.splits["train"]) - g_train)
print(f"Clase minoritaria train = {min_train}")

# Soporte balanceado: N por clase desde train. Meta-training: N_SHOT + N_QUERY por clase desde train.
assert max(N_SHOTS) <= min_train, \
    f"soporte: N_SHOT ({max(N_SHOTS)}) > minoritaria train ({min_train})."
assert max(N_SHOTS) + N_QUERY <= min_train, \
    (f"meta-train: N_SHOT+N_QUERY ({max(N_SHOTS)}+{N_QUERY}) > minoritaria train ({min_train}). "
     f"Baja N_QUERY a <= {min_train - max(N_SHOTS)}.")
print("OK: el barrido entra en train.")

## Celda 3: Barrido de N_SHOT (meta-train + F1 val completo + Grad-CAM)

Por cada seed y cada `N_SHOT`: meta-entreno (n_shot=N_SHOT), armo prototipos de despliegue (soporte
balanceado N-shot de *train*), evalúo F1 en el **val completo**, mido el IoU Grad-CAM, y guardo el `.pth`.

In [ ]:
results = {k: {} for k in N_KEYS}   # results[N_SHOT][seed] = {f1, iou, pointing}

for seed in SEEDS:
    print(f"\n{'='*60}\nseed={seed}\n{'='*60}")
    for nshot in N_SHOTS:
        key = f"N{nshot}"
        # 1) clasificador con backbone ENTRENABLE (meta)
        clf = PrototypeClassifier({"backbone": BACKBONE, "num_classes": 2,
                                   "pretrained": True, "freeze_backbone": False, "seed": seed})
        # 2) meta-training episodico (n_shot = N_SHOT)
        clf.meta_train(data_module, n_way=2, n_shot=nshot, n_query=N_QUERY,
                       n_episodes=N_TRAINING_EPISODES, lr=META_LR, augment=True)
        # 3) prototipos de despliegue: soporte balanceado N-shot de train
        support_ids = build_support_indices(data_module, nshot, seed, "balanced")
        clf.fit(create_few_shot_loader(data_module, support_ids, BATCH_SIZE, seed, augment=False))
        # 4) F1-macro en el VAL COMPLETO (opcion A, igual que la seleccion de backbone)
        f1 = evaluate_classification(clf, val_loader)["f1_macro"]
        # 5) IoU Grad-CAM en el val
        gcm = evaluate_gradcam_quality(clf, val_loader)
        # 6) guardar el modelo
        clf.save(f"{RESULTS_DIR}/{BACKBONE}/meta/seed_{seed}/model_{key}.pth")

        results[key][seed] = {"f1": f1, "iou": gcm["mean_iou"], "pointing": gcm["mean_pointing_accuracy"]}
        print(f"  {key}: F1={f1:.3f}  IoU={gcm['mean_iou']:.3f}  pointing={gcm['mean_pointing_accuracy']:.3f}")
        del clf
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

# VRAM de EfficientNet-B0 (constante para todos los N_SHOT; se mide una vez)
bench_model = CNNClassifier({"backbone": BACKBONE, "num_classes": 2, "pretrained": False, "seed": 42})
vram = run_benchmark(BACKBONE, bench_model, val_loader, device, num_runs=100)
del bench_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"\nVRAM EfficientNet-B0: {vram['vram_batch_1_mb']:.0f} MB (constante entre N_SHOT)")

## Celda 4: Agregar, score por N_SHOT y guardar en Drive

`score = 0.40·F1 + 0.40·IoU + 0.20·(1 − VRAM_norm)`; con un solo backbone `VRAM_norm=0` (constante).
**Mejor N_SHOT = mayor score.**

In [ ]:
summary = {}
for key in N_KEYS:
    f1s  = [results[key][s]["f1"]  for s in SEEDS]
    ious = [results[key][s]["iou"] for s in SEEDS]
    f1_mean, f1_std   = float(np.mean(f1s)),  float(np.std(f1s))
    iou_mean, iou_std = float(np.mean(ious)), float(np.std(ious))
    score = 0.40 * f1_mean + 0.40 * iou_mean + 0.20 * 1.0   # VRAM_norm=0 (un solo backbone)
    summary[key] = {"n_shot": int(key[1:]),
                    "f1_mean": f1_mean, "f1_std": f1_std,
                    "iou_mean": iou_mean, "iou_std": iou_std,
                    "score": score,
                    "per_seed": {str(s): results[key][s] for s in SEEDS}}

best = max(summary, key=lambda k: summary[k]["score"])

print(f"\nRESULTADOS (val completo; mean +/- std sobre {len(SEEDS)} seeds):")
print(f"{'N_SHOT':<8}{'F1':>18}{'IoU':>18}{'Score':>10}")
print("-" * 54)
for key in N_KEYS:
    s = summary[key]
    print(f"{key:<8}{s['f1_mean']:.3f}+-{s['f1_std']:.3f}   {s['iou_mean']:.3f}+-{s['iou_std']:.3f}   {s['score']:.4f}")
print(f"\nMejor N_SHOT: {best} (score={summary[best]['score']:.4f})")

os.makedirs(RESULTS_DIR, exist_ok=True)
final = {
    "experiment": "efficientnet_b0_meta_nshot_fullval",
    "best_n_shot": best,
    "summary": summary,
    "vram_efficientnet_b0_mb": vram["vram_batch_1_mb"],
    "eval": "full val (opcion A)", "n_shots": N_SHOTS, "seeds": SEEDS,
    "n_query_meta": N_QUERY, "n_training_episodes": N_TRAINING_EPISODES, "image_size": IMAGE_SIZE,
    "score_formula": "0.40*F1 + 0.40*IoU + 0.20*(1-VRAM_norm); VRAM_norm=0 (un solo backbone)",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open(f"{RESULTS_DIR}/meta_nshot_summary.json", "w") as f:
    json.dump(final, f, indent=2)
print(f"\nGuardado: {RESULTS_DIR}/meta_nshot_summary.json")
print(f".pth en: {RESULTS_DIR}/{BACKBONE}/meta/seed_<seed>/model_N<n>.pth")